In [1]:
import sqlite3
import os

os.makedirs("data/databases", exist_ok=True)

In [2]:
conn = sqlite3.connect("data/databases/company.db")
cursor = conn.cursor()

#### Create tables

In [3]:
cursor.execute('''create table if not exists employees
               (id integer primary key, name text, role text, department text, salary real)''')

In [4]:
cursor.execute('''CREATE TABLE IF NOT EXISTS projects
                 (id INTEGER PRIMARY KEY, name TEXT, status TEXT, budget REAL, lead_id INTEGER)''')

#### Insert sample data

In [5]:
employees = [
    (1, 'Pavan Pinapatruni', 'AI Engineer', 'Engineering', 150000),
    (2, 'Ramesh Bhimavarapu', 'Mechanical Engineer', 'Engineering', 120000),
    (3, 'Suresh Kumar', 'Data Scientist', 'Data Science', 130000),
    (4, 'Anjali Sharma', 'Product Manager', 'Product', 140000)
]

projects = [
    (1, 'AI Chatbot', 'Active', 500000, 1),
    (2, 'Autonomous Vehicle', 'Planning', 1000000, 2),
    (3, 'Data Analytics Platform', 'Active', 750000, 3),
    (4, 'Mobile App Development', 'Completed', 300000, 4)
]

In [6]:
cursor.executemany('insert or replace into employees values (?, ?, ?, ?, ?)', employees)
cursor.executemany('insert or replace into projects values (?, ?, ?, ?, ?)', projects)

In [7]:
cursor.execute('SELECT * FROM employees')
rows = cursor.fetchall()
for row in rows:
    print(row)

(1, 'Pavan Pinapatruni', 'AI Engineer', 'Engineering', 150000.0)
(2, 'Ramesh Bhimavarapu', 'Mechanical Engineer', 'Engineering', 120000.0)
(3, 'Suresh Kumar', 'Data Scientist', 'Data Science', 130000.0)
(4, 'Anjali Sharma', 'Product Manager', 'Product', 140000.0)


In [8]:
conn.commit()
conn.close()

### Database Content Extraction

In [9]:
from langchain_community.utilities import SQLDatabase
from langchain_community.document_loaders import SQLDatabaseLoader

In [10]:
# Method 1: Using SQLDatabase
db_loader = SQLDatabase.from_uri("sqlite:///data/databases/company.db")

In [11]:
print(f"Tables: {db_loader.get_table_names()}")
print(db_loader.get_table_info())

Tables: ['employees', 'projects']

CREATE TABLE employees (
	id INTEGER, 
	name TEXT, 
	role TEXT, 
	department TEXT, 
	salary REAL, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	role	department	salary
1	Pavan Pinapatruni	AI Engineer	Engineering	150000.0
2	Ramesh Bhimavarapu	Mechanical Engineer	Engineering	120000.0
3	Suresh Kumar	Data Scientist	Data Science	130000.0
*/


CREATE TABLE projects (
	id INTEGER, 
	name TEXT, 
	status TEXT, 
	budget REAL, 
	lead_id INTEGER, 
	PRIMARY KEY (id)
)

/*
3 rows from projects table:
id	name	status	budget	lead_id
1	AI Chatbot	Active	500000.0	1
2	Autonomous Vehicle	Planning	1000000.0	2
3	Data Analytics Platform	Active	750000.0	3
*/


C:\Users\FPK1COB\AppData\Local\Temp\ipykernel_18436\2960190414.py:1: LangChainDeprecationWarning: The method `SQLDatabase.get_table_names` was deprecated in langchain-community 0.0.1 and will be removed in 1.0. Use `get_usable_table_names` instead.
  print(f"Tables: {db_loader.get_table_names()}")


In [18]:
# Method 2: Custom SQL to Document Conversion

from typing import List
from langchain_core.documents import Document


def sql_to_documents(db_path:str) -> List[Document]:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    documents = []

    # Strategy1 :  Create Documents for each table
    cursor.execute("select name from sqlite_master where type='table';")
    tables = cursor.fetchall()

    for table in tables:
        table_name = table[0]

        # Get table schema
        cursor.execute(f"PRAGMA table_info({table_name});")
        columns = cursor.fetchall()
        column_names = [col[1] for col in columns]

        #Get table data
        cursor.execute(f"select * from {table_name};")
        rows = cursor.fetchall()

        # Create table overview doc
        content = f"Table: {table_name}\n"
        content += f"Columns: {', '.join(column_names)}\n"
        content += f"Total Records: {len(rows)}\n\n"

        # Add sample records
        content += "Sample Records:\n"
        for row in rows:
            record = dict(zip(column_names, row))
            content += f"{record}\n"

        doc = Document(
            page_content=content,
            metadata={
                'source':db_path,
                'table_name':table_name,
                'num_records':len(rows),
                'data_type':'sql_table'
            }
        )
        documents.append(doc)

        # Strategy2: Create a relationship documents
        # Example : Join employees and projects
        cursor.execute("""
        SELECT e.name, e.role, p.name as project_name, p.status
        FROM employees e
        JOIN projects p ON e.id = p.lead_id;
        """)

        relationships = cursor.fetchall()
        rel_content = "Employee-Project Relationships:\n"
        for rel in relationships:
            rel_content += f"Employee: {rel[0]}, Role: {rel[1]}, Project: {rel[2]}, Status: {rel[3]}\n"

        rel_doc = Document(
            page_content=rel_content,
            metadata={
                'source':db_path,
                'data_type':'sql_relationship',
                'query':'employee-project join'
            }
        )
        documents.append(rel_doc)

        conn.close()
        return documents

In [19]:
sql_to_documents("data/databases/company.db")

[Document(metadata={'source': 'data/databases/company.db', 'table_name': 'employees', 'num_records': 4, 'data_type': 'sql_table'}, page_content="Table: employees\nColumns: id, name, role, department, salary\nTotal Records: 4\n\nSample Records:\n{'id': 1, 'name': 'Pavan Pinapatruni', 'role': 'AI Engineer', 'department': 'Engineering', 'salary': 150000.0}\n{'id': 2, 'name': 'Ramesh Bhimavarapu', 'role': 'Mechanical Engineer', 'department': 'Engineering', 'salary': 120000.0}\n{'id': 3, 'name': 'Suresh Kumar', 'role': 'Data Scientist', 'department': 'Data Science', 'salary': 130000.0}\n{'id': 4, 'name': 'Anjali Sharma', 'role': 'Product Manager', 'department': 'Product', 'salary': 140000.0}\n"),
 Document(metadata={'source': 'data/databases/company.db', 'data_type': 'sql_relationship', 'query': 'employee-project join'}, page_content='Employee-Project Relationships:\nEmployee: Pavan Pinapatruni, Role: AI Engineer, Project: AI Chatbot, Status: Active\nEmployee: Ramesh Bhimavarapu, Role: Mech